In [ ]:
import numpy as np
import tempfile, os
from dnb import Pipeline, FileSource, PipelineConfig, EventType
from dnb.modules import WaveletConvolution, EventDetector, PowerEstimator

# Make a 1-second fake recording with a 120Hz burst on channel 0
rng = np.random.default_rng(42)
n, fs = 30000, 30000.0
t = np.arange(n) / fs
data = rng.standard_normal((4, n)) * 0.5
data[0, 12000:18000] += 10 * np.sin(2 * np.pi * 120 * t[12000:18000])

path = os.path.join(tempfile.mkdtemp(), "test.npz")
np.savez(path, continuous=data, sample_rate=fs)

events = Pipeline(
    source=FileSource(path),
    modules=[
        WaveletConvolution(freq_min=10, freq_max=200, n_freqs=15),
        PowerEstimator(),
        EventDetector(event_type=EventType.RIPPLE, freq_range=(80, 200), threshold_std=3.0),
    ],
    config=PipelineConfig(sample_rate=fs, n_channels=4, chunk_duration=0.2),
).run_offline()

print(f"Detected {len(events)} events")
for e in events[:5]:
    print(f"  {e.event_type.name} ch={e.channel_id} t={e.timestamp:.3f}s dur={e.duration:.3f}s")